# Tier 2: 750-Step Proof (~56 min)

**Purpose**: Catch late bloomers. Verify Tier 1 winners aren't flukes.

Only run this if the mutation **passed Tier 1** (step-250 BPB < 2.04).

| Decision | Condition |
|----------|----------|
| ❌ Kill | step-750 BPB **> 1.75** |
| ⚠️ Marginal | step-750 BPB **1.70 – 1.75** |
| ✅ Promote to Tier 3 | step-750 BPB **< 1.70** |

**Baseline reference**: 1.7214 BPB @ step 750 (T4, 32k batch, seed 42)

### Why 750 catches late bloomers
At step 750 the model's capacity ceiling is visible but the warmdown phase
(step 2500+) hasn't started. If an architecture tracks below baseline here,
the warmdown will push it further ahead. If it's above, warmdown can't save it.

In [ ]:
# [CELL 1] CLONE
%cd /content
!rm -rf uniform-int4
!git clone https://github.com/jmoncayo-pursuit/parameter-golf-uniform-int4.git uniform-int4
print('✅ Repo cloned')

In [ ]:
# [CELL 2] ASSETS
%cd /content/uniform-int4
!mkdir -p data/tokenizers data/datasets/fineweb10B_sp1024
!wget -q -O data/tokenizers/fineweb_1024_bpe.model \
  https://huggingface.co/datasets/willdepueoai/parameter-golf/resolve/main/datasets/tokenizers/fineweb_1024_bpe.model
!wget -q -O data/datasets/fineweb10B_sp1024/fineweb_val_000000.bin \
  https://huggingface.co/datasets/willdepueoai/parameter-golf/resolve/main/datasets/datasets/fineweb10B_sp1024/fineweb_val_000000.bin
!wget -q -O data/datasets/fineweb10B_sp1024/fineweb_train_000001.bin \
  https://huggingface.co/datasets/willdepueoai/parameter-golf/resolve/main/datasets/datasets/fineweb10B_sp1024/fineweb_train_000001.bin
!pip install -q sentencepiece zstandard PyYAML huggingface-hub datasets tqdm

import os
assert os.path.getsize('data/tokenizers/fineweb_1024_bpe.model') > 100_000
assert os.path.getsize('data/datasets/fineweb10B_sp1024/fineweb_val_000000.bin') > 1_000_000
assert os.path.getsize('data/datasets/fineweb10B_sp1024/fineweb_train_000001.bin') > 100_000_000
print('✅ All assets verified')

In [ ]:
# [CELL 3] 750-STEP PROOF
%cd /content/uniform-int4
!env NO_COMPILE=1 ITERATIONS=750 VAL_LOSS_EVERY=250 python3 train_gpt.py

In [ ]:
# [CELL 4] VERDICT
bpb = float(input('Enter step-750 val_bpb: '))

BASELINE = 1.7214
if bpb > 1.75:
    print(f'❌ KILL — {bpb:.4f} won\'t reach leaderboard. Warmdown can\'t recover this gap.')
elif bpb > BASELINE:
    print(f'⚠️ MARGINAL — {bpb:.4f} is close to baseline ({BASELINE}). Could be a late bloomer. Consider Tier 3 with caution.')
elif bpb > 1.70:
    print(f'🔶 STRONG — {bpb:.4f} beats baseline ({BASELINE}). Strong candidate for Tier 3 full proof.')
else:
    print(f'✅ PROMOTE — {bpb:.4f} significantly beats baseline ({BASELINE}). Run Tier 3 full proof.')